In [ ]:
# ============================================================
# CÉLULA 0 — PREPARAÇÃO DO AMBIENTE
# ============================================================
# No Google Colab, o ipywidgets já vem instalado.
# O Lark precisa ser instalado (leva ~5 segundos).
# No VS Code, rode no terminal: pip install lark ipywidgets ipykernel
# No Google Colab insira isto antes dos imports: %pip install -q lark ipywidgets
# ------------------------------------------------------------

import html
import lark
from lark import Lark
from lark.exceptions import UnexpectedCharacters, UnexpectedInput
import ipywidgets as widgets
from IPython.display import display, HTML

print(f" Lark versão {lark.__version__} pronto!")
print(f" ipywidgets versão {widgets.__version__} pronto!")

 Lark versão 1.3.1 pronto!
 ipywidgets versão 8.1.9 pronto!


In [4]:
# ============================================================
# CÉLULA 1 — FERRAMENTAS DE VISUALIZAÇÃO (usadas por todos os exemplos)
# ============================================================
# Estas funções NÃO fazem análise léxica. Elas só "desenham"
# os tokens de forma bonita (tabela e texto colorido).

# Paleta de cores: cada categoria de token ganha uma cor.
CORES = {
    # --- Palavras reservadas ---
    "INGRESSO": "#1565c0",
    "SETOR": "#1565c0",
    "LOTE": "#1565c0",
    "MEIA": "#1565c0",
    "INTEIRA": "#1565c0",
    "DATA": "#1565c0",

    # --- Literais ---
    "QTD": "#ef6c00",
    "NUMERO": "#ef6c00",
    "VALOR": "#2e7d32",
    "DATA_VAL": "#4e342e",
    "TEXTO": "#c62828",
    "SETOR_NOME": "#00838f",

    # --- Símbolos ---
    "SIMBOLO_X": "#6a1b9a",

    # --- Comentários ---
    "COMENTARIO": "#757575",
}

def cor_do_token(tipo):
    """Devolve a cor da categoria (cinza se não estiver na paleta)."""
    return CORES.get(tipo, "#455a64")

def tabela_tokens_html(tokens, titulo="Tabela de Tokens"):
    """Monta uma tabela HTML com: nº, TOKEN (categoria), LEXEMA, linha e coluna."""
    linhas = ""
    for i, t in enumerate(tokens, start=1):
        cor = cor_do_token(t.type)
        linhas += (
            f"<tr><td>{i}</td>"
            f"<td><b style='color:{cor}'>{t.type}</b></td>"
            f"<td><code>{html.escape(str(t.value))}</code></td>"
            f"<td>{t.line}</td><td>{t.column}</td></tr>"
        )
    return f"""
    <h4 style='margin:4px 0'>{titulo} ({len(tokens)} tokens)</h4>
    <table style='border-collapse:collapse;font-family:monospace;font-size:13px'>
      <tr style='background:#263238;color:white'>
        <th style='padding:4px 10px'>#</th><th style='padding:4px 10px'>TOKEN</th>
        <th style='padding:4px 10px'>LEXEMA</th><th style='padding:4px 10px'>LINHA</th>
        <th style='padding:4px 10px'>COLUNA</th>
      </tr>
      {linhas}
    </table>"""

def texto_colorido_html(texto, tokens, transformar=None):
    """Pinta cada lexema no texto original, como faz o 'syntax highlight' do VS Code.
    Usa t.start_pos e t.end_pos: a posição exata de cada token no texto.
    'transformar' (opcional) troca o lexema exibido — usado para mascarar dados (LGPD)."""
    saida, cursor = "", 0
    for t in tokens:
        saida += html.escape(texto[cursor:t.start_pos])        # espaços/comentários
        cor = cor_do_token(t.type)
        saida += (f"<span title='{t.type}' style='color:{cor};font-weight:bold;"
                  f"border-bottom:2px solid {cor}'>"
                  f"{html.escape(transformar(t) if transformar else texto[t.start_pos:t.end_pos])}</span>")
        cursor = t.end_pos
    saida += html.escape(texto[cursor:])
    return ("<pre style='background:#fafafa;border:1px solid #ddd;padding:10px;"
            f"font-size:14px;line-height:1.7'>{saida}</pre>"
            "<small>💡 Passe o mouse sobre um lexema para ver o nome do token.</small>")

def erro_lexico_html(texto, erro):
    """Mostra um erro léxico com linha, coluna e uma 'setinha' apontando o problema."""
    linha_txt = texto.splitlines()[erro.line - 1] if texto.splitlines() else ""
    seta = " " * (erro.column - 1) + "^"
    return f"""
    <div style='background:#ffebee;border-left:5px solid #c62828;padding:10px'>
      <b>❌ ERRO LÉXICO</b> na linha <b>{erro.line}</b>, coluna <b>{erro.column}</b>:
      caractere inesperado <code>{html.escape(repr(erro.char))}</code>
      <pre style='margin:6px 0'>{html.escape(linha_txt)}\n{seta}</pre>
      <small>{html.escape(getattr(erro, 'dica', ''))}</small>
    </div>"""

print("✅ Ferramentas de visualização carregadas!")

✅ Ferramentas de visualização carregadas!


In [5]:
# ============================================================
# CÉLULA 2 — ANALISADOR LÉXICO DE INGRESSOS
# Cenário: uma plataforma de eventos recebe uma linha com
# informações sobre um ingresso e precisa identificar cada
# elemento da entrada.
# Foco: palavras reservadas, literais, prioridade e ambiguidade.
# ============================================================

gramatica_ingresso = r"""
// ============================================================
// PALAVRAS RESERVADAS
// ============================================================

INGRESSO.3: /\bINGRESSO\b/i
SETOR.3:     /\bSETOR\b/i
LOTE.3:      /\bLOTE\b/i
MEIA.3:      /\bMEIA\b/i
INTEIRA.3:   /\bINTEIRA\b/i
DATA.3:      /\bDATA\b/i

// ============================================================
// LITERAIS
// ============================================================

// Quantidade de ingressos.
// Exemplo: 2x
QTD.4: /\d+x\b/i

// Número inteiro.
// Exemplo: 2
NUMERO.2: /\d+/

// Valor em reais.
// Exemplos: R$ 180,00 ou R$ 1.250,00
VALOR.2: /R\$ ?\d{1,3}(\.\d{3})*,\d{2}/

// Data do evento.
// Exemplo: 20/07/2026
DATA_VAL.4: /\d{2}\/\d{2}\/\d{4}/

// Nome do evento entre aspas.
// Exemplo: "Festival de Inverno"
TEXTO.2: /"[^"\n]+"/

// Nome do setor.
// Exemplos: pista, camarote, arquibancada
SETOR_NOME.1: /[A-Za-zÀ-ÿ]+/

// ============================================================
// COMENTÁRIOS E ESPAÇOS
// ============================================================

COMMENT: /#[^\n]*/
%ignore COMMENT
%ignore /[ \t\r\n]+/
"""

# Como este trabalho é um ANALISADOR LÉXICO,
# usamos o lexer diretamente, sem parser sintático.
lexer_ingresso = Lark(
    gramatica_ingresso,
    parser=None,
    lexer="basic",
    propagate_positions=True
)

def tokenizar_ingresso(texto):
    return list(lexer_ingresso.lex(texto))


# ============================================================
# TESTE INICIAL
# ============================================================

linha = 'INGRESSO 2x "Festival de Inverno" SETOR pista LOTE 2 MEIA R$ 180,00 DATA 20/07/2026'

for t in tokenizar_ingresso(linha):
    print(f"{t.type:<15} {t.value!r}")

INGRESSO        'INGRESSO'
QTD             '2x'
TEXTO           '"Festival de Inverno"'
SETOR           'SETOR'
SETOR_NOME      'pista'
LOTE            'LOTE'
NUMERO          '2'
MEIA            'MEIA'
VALOR           'R$ 180,00'
DATA            'DATA'
DATA_VAL        '20/07/2026'


In [ ]:
# ============================================================
# CÉLULA 3 — INTERFACE + LABORATÓRIO DE PRIORIDADE
# Escolha um ingresso e altere a prioridade da DATA_VAL
# para observar o efeito da prioridade léxica.
# ============================================================

exemplos_ingresso = {
    "🎫 Festival de Inverno":
        'INGRESSO 2x "Festival de Inverno" SETOR pista LOTE 2 MEIA R$ 180,00 DATA 20/07/2026',

    "🎸 Rock em São Paulo":
        'INGRESSO 1x "Rock em São Paulo" SETOR pista LOTE 3 INTEIRA R$ 250,00 DATA 15/08/2026',

    "🎵 Festival de Música":
        'INGRESSO 4x "Festival de Música" SETOR camarote LOTE 1 MEIA R$ 1.250,00 DATA 10/09/2026',

    "🎤 Show Nacional":
        'ingresso 2x "Show Nacional" setor arquibancada lote 2 inteira R$ 320,00 data 05/10/2026',

    "🧪 Teste com comentário":
        'INGRESSO 2x "Festival de Inverno" SETOR pista LOTE 2 MEIA R$ 180,00 DATA 20/07/2026 # compra antecipada',

    "❌ Data problemática":
        'INGRESSO 2x "Festival de Inverno" SETOR pista LOTE 2 MEIA R$ 180,00 DATA 20/07/26'
}


# ============================================================
# CONTROLES DA INTERFACE
# ============================================================

seletor_ingresso = widgets.Dropdown(
    options=list(exemplos_ingresso),
    description="Exemplos:",
    layout=widgets.Layout(width="70%")
)

entrada_ingresso = widgets.Text(
    value=exemplos_ingresso["🎫 Festival de Inverno"],
    description="Entrada:",
    layout=widgets.Layout(width="95%")
)

prioridade_data = widgets.IntSlider(
    value=4,
    min=1,
    max=5,
    description="Prior. data:",
    style={"description_width": "initial"}
)

explica_ingresso = widgets.HTML(
    "<small>"
    "A regra <code>DATA_VAL</code> compete com <code>NUMERO</code> "
    "quando encontra uma data como <code>20/07/2026</code>. "
    "Com prioridade maior, <code>DATA_VAL</code> reconhece a data inteira. "
    "Com prioridade menor ou igual à de <code>NUMERO</code>, o lexer pode "
    "reconhecer primeiro apenas <code>20</code>."
    "</small>"
)

botao_ingresso = widgets.Button(
    description="🔍 Analisar ingresso",
    button_style="primary"
)

saida_ingresso = widgets.Output()


# ============================================================
# FUNÇÃO PRINCIPAL
# ============================================================

def ao_clicar_ingresso(_):

    saida_ingresso.clear_output()

    # Recria a gramática utilizando a prioridade escolhida
    # no slider.
    gramatica = gramatica_ingresso.replace(
        "DATA_VAL.4",
        f"DATA_VAL.{prioridade_data.value}"
    )

    lexer = Lark(
        gramatica,
        parser=None,
        lexer="basic",
        propagate_positions=True
    )

    texto = entrada_ingresso.value

    with saida_ingresso:

        try:
            tokens = list(lexer.lex(texto))

            # -----------------------------------------------
            # Texto colorido
            # -----------------------------------------------

            display(
                HTML(
                    texto_colorido_html(
                        texto,
                        tokens
                    )
                )
            )

            # -----------------------------------------------
            # Tabela de tokens
            # -----------------------------------------------

            display(
                HTML(
                    tabela_tokens_html(
                        tokens
                    )
                )
            )

        except UnexpectedCharacters as erro:

            # Dicas específicas do domínio de ingressos
            if erro.char == '"':
                erro.dica = (
                    "Dica: o nome do evento deve estar entre aspas, "
                    'por exemplo: "Festival de Inverno".'
                )

            elif erro.char == '/':
                erro.dica = (
                    "Dica: a data do evento deve usar o formato "
                    "DD/MM/AAAA, por exemplo: 20/07/2026."
                )

            elif erro.char == '$':
                erro.dica = (
                    "Dica: o preço deve começar com R$ e possuir "
                    "duas casas decimais, por exemplo: R$ 180,00."
                )

            else:
                erro.dica = (
                    "Dica: verifique se o ingresso utiliza o formato "
                    "esperado, como quantidade, evento, setor, lote, "
                    "tipo, preço e data."
                )

            display(
                HTML(
                    erro_lexico_html(
                        texto,
                        erro
                    )
                )
            )


# ============================================================
# SELETOR DE EXEMPLOS
# ============================================================

def ao_escolher_ingresso(m):

    entrada_ingresso.value = exemplos_ingresso[m["new"]]

    ao_clicar_ingresso(None)


# ============================================================
# EVENTOS DOS CONTROLES
# ============================================================

seletor_ingresso.observe(
    ao_escolher_ingresso,
    names="value"
)

prioridade_data.observe(
    lambda m: ao_clicar_ingresso(None),
    names="value"
)

botao_ingresso.on_click(
    ao_clicar_ingresso
)


# ============================================================
# EXIBIÇÃO DA INTERFACE
# ============================================================

display(
    widgets.HTML(
        "<h3>🎫 Analisador Léxico de Ingressos</h3>"
    ),

    seletor_ingresso,

    entrada_ingresso,

    widgets.HBox([
        prioridade_data,
        botao_ingresso
    ]),

    explica_ingresso,

    saida_ingresso
)


# Executa a análise inicial
ao_clicar_ingresso(None)

HTML(value='<h3>🎫 Analisador Léxico de Ingressos</h3>')

Dropdown(description='Exemplos:', layout=Layout(width='70%'), options=('🎫 Festival de Inverno', '🎸 Rock em São…

Text(value='INGRESSO 2x "Festival de Inverno" SETOR pista LOTE 2 MEIA R$ 180,00 DATA 20/07/2026', description=…

HTML(value='<small>A regra <code>DATA_VAL</code> compete com <code>NUMERO</code> quando encontra uma data como…

Output()